In [0]:
print("Ask Sarthak environment setup")

print("Spark version:", spark.version)

display(
    spark.sql("""
        SELECT
            current_catalog(),
            current_schema()
    """)
)

#Create Catalog
spark.sql("""
CREATE CATALOG IF NOT EXISTS ask_sarthak
""")

#Create Schemas
schemas = [
    "bronze",
    "silver",
    "gold",
    "analytics"
]

for schema in schemas:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS
        ask_sarthak.{schema}
    """)

print("All Ask Sarthak schemas created successfully!")

# show schemas
display(
    spark.sql("""
        SHOW SCHEMAS IN ask_sarthak
    """)
)

#Create Volume
spark.sql("""
CREATE VOLUME IF NOT EXISTS
ask_sarthak.bronze.finance_source_files
""")

display(
    spark.sql("""
        SHOW VOLUMES IN ask_sarthak.bronze
    """)
)

#Create 500000 journal entries

from pyspark.sql import functions as F

# Generate 500,000 journal entries
entries = (
    spark.range(1, 500001)
    .withColumnRenamed("id", "entry_id")
    .withColumn(
        "company_code",
        F.element_at(
            F.array(
                F.lit("1000"), F.lit("2000"), F.lit("3000"), F.lit("4000"), F.lit("5000")
            ),
            (F.pmod("entry_id", F.lit(5)) + 1).cast("int")
        )
    )
    .withColumn(
        "posting_date",
        F.date_add(
            F.to_date(F.lit("2024-01-01")),
            F.pmod("entry_id", F.lit(1040)).cast("int")
        )
    )
    .withColumn("document_date", F.col("posting_date"))
    .withColumn("fiscal_year", F.year("posting_date"))
    .withColumn(
        "transaction_type",
        F.pmod("entry_id", F.lit(4)).cast("int")
    )
    .withColumn(
        "amount",
        F.round(
            (F.pmod(F.col("entry_id") * 7919, F.lit(4990000)) + 10000) / 100,
            2
        )
    )
    .withColumn("currency", F.lit("INR"))
)

print("Journal entries generated")

# Debit side
debit = (
    entries
    .withColumn("line_item", F.lit("001"))
    .withColumn("debit_credit", F.lit("D"))
    .withColumn(
        "gl_account",
        F.when(F.col("transaction_type") == 0, "500100")
         .when(F.col("transaction_type") == 1, "100100")
         .when(F.col("transaction_type") == 2, "150100")
         .otherwise("200100")
    )
)

# Credit side
credit = (
    entries
    .withColumn("line_item", F.lit("002"))
    .withColumn("debit_credit", F.lit("C"))
    .withColumn(
        "gl_account",
        F.when(F.col("transaction_type") == 1, "400100")
         .otherwise("100100")
    )
)

# Combine debit and credit
journal_lines = debit.unionByName(credit)

journal_lines = (
    journal_lines
    .withColumn(
        "account_name",
        F.when(F.col("gl_account") == "100100", "Bank Account")
         .when(F.col("gl_account") == "150100", "Equipment")
         .when(F.col("gl_account") == "200100", "Accounts Payable")
         .when(F.col("gl_account") == "400100", "Sales Revenue")
         .otherwise("Rent Expense")
    )
    .withColumn(
        "account_type",
        F.when(F.col("gl_account").startswith("1"), "ASSET")
         .when(F.col("gl_account").startswith("2"), "LIABILITY")
         .when(F.col("gl_account").startswith("4"), "REVENUE")
         .otherwise("EXPENSE")
    )
    .withColumn(
        "document_number",
        F.concat(
            F.lit("FI"),
            F.lpad(F.col("entry_id").cast("string"), 10, "0")
        )
    )
    .withColumn(
        "document_type",
        F.when(F.col("transaction_type") == 0, "EXPENSE")
         .when(F.col("transaction_type") == 1, "SALES")
         .when(F.col("transaction_type") == 2, "ASSET_PURCHASE")
         .otherwise("VENDOR_PAYMENT")
    )
    .withColumn(
        "vendor_id",
        F.when(
            F.col("transaction_type").isin(0, 2, 3),
            F.concat(
                F.lit("V"),
                F.lpad(
                    (F.pmod("entry_id", F.lit(1000)) + 1).cast("string"),
                    4, "0"
                )
            )
        )
    )
    .withColumn(
        "customer_id",
        F.when(
            F.col("transaction_type") == 1,
            F.concat(
                F.lit("C"),
                F.lpad(
                    (F.pmod("entry_id", F.lit(1000)) + 1).cast("string"),
                    4, "0"
                )
            )
        )
    )
    .withColumn(
        "cost_center",
        F.concat(
            F.lit("CC"),
            F.lpad(
                (F.pmod("entry_id", F.lit(100)) + 1).cast("string"),
                3, "0"
            )
        )
    )
    .withColumn(
        "profit_center",
        F.concat(
            F.lit("PC"),
            F.lpad(
                (F.pmod("entry_id", F.lit(20)) + 1).cast("string"),
                3, "0"
            )
        )
    )
    .withColumn(
        "reference",
        F.concat(F.lit("REF-"), F.col("entry_id"))
    )
    .withColumn(
        "description",
        F.concat(F.col("document_type"), F.lit(" transaction"))
    )
    .select(
        "company_code",
        "entry_id",
        "fiscal_year",
        "document_number",
        "line_item",
        "posting_date",
        "document_date",
        "gl_account",
        "account_name",
        "account_type",
        "debit_credit",
        "amount",
        "currency",
        "vendor_id",
        "customer_id",
        "cost_center",
        "profit_center",
        "document_type",
        "reference",
        "description"
    )
)

#validation of journals
print("Journal line items prepared")


print("Total journal line items:", journal_lines.count())

display(journal_lines.limit(10))

#Check debit and credit balances
display(
    journal_lines
    .groupBy("debit_credit")
    .agg(
        F.count("*").alias("record_count"),
        F.sum("amount").alias("total_amount")
    )
)

#save the file (write it to unity catalog)

source_path = (
    "/Volumes/ask_sarthak/bronze/"
    "finance_source_files/journal_lines"
)

(
    journal_lines
    .repartition(8)
    .write
    .format("csv")
    .option("header", "true")
    .mode("overwrite")
    .save(source_path)
)

print("Finance source files saved successfully")

#verify the saved data

saved_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

print("Saved records:", saved_df.count())

display(saved_df.limit(10))